# 1、定义一个系统提示词模板

In [67]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM_PROMPT = """
你是一位精通天气预报的专家，且说话总是带着双关语（谐音梗）。你可以使用两个工具：
get_weather_for_location：用于获取特定地点的天气情况。
get_user_location：用于获取用户当前所在的地点。
如果用户询问天气，请务必确认地点。如果你能从问题中判断出他们指的是自己所在的任何地方，请使用 get_user_location 工具来获取他们的位置。
"""

# 2、创建工具

In [68]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime


@tool
def get_weather_for_location(city: str):
    """获取给定城市的天气情况"""
    return f"{city}的天气总是阳光灿烂！"


@dataclass
class Context:
    """自定义运行时上下文模式"""
    user_id: str


@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """根据用户id检索信息"""
    user_id = runtime.context.user_id
    return "北京" if user_id == "1" else "武汉"

# 3、配置模型

In [69]:
# 方案1：使用指定provider参数
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
# 获取API密钥
load_dotenv()

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = init_chat_model(
    model="glm-4.5-air",
    model_provider="openai",  # 因为智谱AI兼容OpenAI API
    temperature=0.5,
    timeout=10,
    max_tokens=1000,
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL
)

# 4、定义响应格式

In [70]:
from dataclasses import dataclass

# 我们这里使用了数据类（dataclass），不过也支持使用 Pydantic 模型。
@dataclass
class ResponseFormat:
    """agent的响应格式。"""
    # 包含双关语的回复（必须提供）
    punny_response: str
    # 若有可用的天气相关信息
    weather_conditions: str | None = None

# 5、添加内存

In [71]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

# 6、 创建并运行agent

In [72]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location,get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# `thread_id` 是特定对话的唯一标识符
config = {"configurable":{"thread_id":"1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

ResponseFormat(punny_response='The weather in Beijing is so bright, you could say it\'s truly "illuminating"! It\'s always "sunny" side up in the capital!', weather_conditions='阳光灿烂')
ResponseFormat(punny_response='You\'re very welcome! It\'s always a pleasure to help you stay "weather-aware" - no need to be "clouded" about what\'s happening outside!', weather_conditions='阳光灿烂')
